# 03 · The remote-work metric

The paper's metric is `flag_remote_work_smoothed_max_thresh`: for each device and month,
the **share of working days that were remote days**. Full specification:
`notebook_docs/methodology/REMOTE_WORK_FLAGS.md`. In short:

1. Every dwell is labelled `home`, `work` or `other` (within `location_radius_meters`
   of the detected home / work geohash) and exploded into clock hours.
2. **Office days** are workdays with at least one hour at work. On office days we learn,
   per device and clock hour, how likely the device is at home / work / elsewhere
   (Bayes: P(location | hour) from P(hour | location) and P(location)).
3. An hour on a *non-office* workday counts as **remote work** when the device is at home,
   the hour is a working hour (`working_hour_start`–`working_hour_end`) and its office-day
   profile says "work" for that hour. The flag is weighted by the share of all devices at
   work in that hour (smoothing).
4. A day is a **remote day** when its maximum smoothed flag exceeds `smoothed_day_threshold`;
   the metric is remote days / working days.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from remote_work_utils.pipeline import STEPS, run_month
from remote_work_utils.plotting import mean_by
from remote_work_utils.remote_work import SELECTED_METRIC
from remote_work_utils.remote_work.aggregate import (
    filter_user_months, geohash_area_lookup, load_monthly_outputs, quarterly_best_month, stat_area_monthly,
)
from remote_work_utils.remote_work.metrics import general_location_shares
from remote_work_utils.settings import load_settings, resolve_path
from remote_work_utils.validation import homes_vs_population

settings = load_settings("../configs/demo.json")
months = settings["aggregation"]["months"]
truth = pd.read_csv(resolve_path(settings, "synthetic_users"))

## Run all steps for every month
The same as `python pipelines/run_month.py --config configs/demo.json --month 201911 202004 202110 --steps prepare home work remote`.

In [ ]:
results = {month: run_month(settings, month, steps=STEPS) for month in months}
pd.DataFrame({m: r.metrics for m, r in results.items()}).T[["num_homes", "num_works", "num_remote_work_users"]]

## Where devices are, hour by hour, on office days
Step 2 above: the share of all devices at home / work / elsewhere by clock hour on office days.
These shares smooth the individual flags.

In [ ]:
shares = general_location_shares(results["201911"].hours) * 100

fig, ax = plt.subplots(figsize=(7, 4))
for column, color in [("home", "#2b6cb0"), ("work", "#c53030"), ("other", "#718096")]:
    ax.plot(shares.index, shares[column], marker="o", ms=3, color=color, label=column)
ax.axvspan(settings["remote_work"]["working_hour_start"], settings["remote_work"]["working_hour_end"],
           color="#fefcbf", alpha=0.5, label="Working hours")
ax.set_title("Location of devices by hour on office days, 201911")
ax.set_xlabel("Hour of day (local time)")
ax.set_ylabel("Share of device-hours (%)")
ax.legend()
plt.show()

> **Fig. 3.1.** Share of device-hours at home, at work and elsewhere, by hour of day, on office days.
> Work dominates from 9:00 to 17:00. *Notes:* synthetic data; office days are workdays with
> at least one hour at the detected work location.

## The metric, per month
User-months with fewer than `aggregation.min_total_work_days` working days are dropped.
For devices with an office, the synthetic truth is the expected share of remote days,
`min(max_remote_day_probability, remote_day_probability[month] × affinity)`.

In [ ]:
user_months = filter_user_months(load_monthly_outputs(settings, "remote_work_users", months), settings)
user_months = user_months.merge(truth[["identifier", "has_office", "remote_affinity"]], on="identifier")
office_users = user_months[user_months["has_office"]].copy()

synthetic = settings["synthetic"]
office_users["true_share"] = np.minimum(
    synthetic["max_remote_day_probability"],
    office_users["month"].map(synthetic["remote_day_probability"]) * office_users["remote_affinity"],
)
estimated = mean_by(office_users, ["month"], SELECTED_METRIC)
expected = office_users.groupby("month")["true_share"].mean()

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(estimated["month"], estimated["mean"] * 100,
            yerr=[(estimated["mean"] - estimated["ci_low"]) * 100, (estimated["ci_high"] - estimated["mean"]) * 100],
            fmt="o-", capsize=3, color="#2b6cb0", label="Estimated (mean ± 95% CI)")
ax.plot(expected.index, expected.values * 100, "s--", color="#dd6b20", label="Synthetic truth")
ax.set_title("Share of working days worked remotely, devices with an office")
ax.set_xlabel("Month (YYYYMM)")
ax.set_ylabel("Remote days / working days (%)")
ax.legend()
plt.show()
estimated.assign(truth=expected.values).round(3)

> **Fig. 3.2.** Monthly remote-work metric for devices with an office, against the synthetic truth.
> The method tracks the rise in April 2020 and the partial return in October 2021; it
> underestimates the peak because remote days spent at third places are not counted as remote
> work at home. *Notes:* mean over user-months with ≥ 7 working days; 95% normal CIs.

## Statistical areas
Residents are assigned to the statistical area containing their home geohash's centroid.
Area means rest on few devices, so only rows with at least `aggregation.min_homes_per_area`
homes are published, and the quarterly panel keeps, per quarter, the month with enough homes
(most homes on ties).

In [ ]:
stat_areas = gpd.read_file(resolve_path(settings, "stat_areas"))
homes = load_monthly_outputs(settings, "homes", months)
lookup = geohash_area_lookup(homes["geohash"], stat_areas, settings)
area_months = stat_area_monthly(user_months, homes, lookup, settings)
quarterly = quarterly_best_month(area_months, settings)
quarterly.head()

In [ ]:
area_id = settings["stat_areas"]["id_column"]
minimum = settings["aggregation"]["min_homes_per_area"]
fig, axes = plt.subplots(1, len(months), figsize=(13, 5), sharey=True)
for ax, month in zip(axes, months):
    data = stat_areas.merge(area_months[(area_months["month"] == month) & (area_months["num_homes"] >= minimum)], on=area_id)
    stat_areas.plot(ax=ax, color="#edf2f7", edgecolor="white")
    data.plot(ax=ax, column="WFH_mean", cmap="viridis", vmin=0, vmax=0.8, edgecolor="white")
    ax.set_title(month)
    ax.set_xlabel("Longitude (°)")
axes[0].set_ylabel("Latitude (°)")
fig.colorbar(axes[-1].collections[-1], ax=axes, shrink=0.7, label="Remote days / working days")
fig.suptitle("Remote-work metric of residents by statistical area")
plt.show()

> **Fig. 3.3.** Mean remote-work metric of residents by statistical area and month.
> *Notes:* synthetic data; grey areas have fewer than three resident devices and are suppressed.

## Validation: are detected homes representative?
In the paper the same check uses the 2022 census (`homes_vs_population`), and the remote-work
trend is compared with public-transport ticket validations (`wfh_vs_series`).

In [ ]:
population = settings["stat_areas"]["population_column"]
homes_per_area = homes.merge(lookup, on="geohash").groupby([area_id, "month"]).size().rename("num_homes").reset_index()
homes_vs_population(homes_per_area, stat_areas, area_id, population).round(3)